# 02. Pretrained Remote-Sensing Baseline: Tier 1 & Tier 2 Evaluation

This notebook establishes the experimental baseline for 12-band Sentinel-2 land-cover classification using a two-tier strategy:
* **Tier 1**: Class-frequency statistical baseline (predicting marginal label distribution).
* **Tier 2**: Frozen 12-band SeCo ResNet-50 backbone with a trained linear probe head (`Linear(2048, 19)`).

## Section 1: Tier 1 Class-Frequency Statistical Baseline
We compute the empirical label frequencies from the training split and evaluate the baseline metrics.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader

repo_root = Path.cwd().parent.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from ml.landcover.src.dataset import BigEarthNetDataset, create_synthetic_bigearthnet_samples
from ml.landcover.src.evaluate import compute_tier1_frequency_baseline, compute_multilabel_metrics
from ml.landcover.src.model import LandCoverClassifier
from ml.landcover.src.train import train_tier2_linear_probe

# Prepare train and validation data
train_dir = repo_root / 'ml' / 'landcover' / 'data' / 'train_samples'
val_dir = repo_root / 'ml' / 'landcover' / 'data' / 'val_samples'
train_samples = create_synthetic_bigearthnet_samples(str(train_dir), num_samples=60)
val_samples = create_synthetic_bigearthnet_samples(str(val_dir), num_samples=30)

train_dataset = BigEarthNetDataset(train_samples, split='train')
val_dataset = BigEarthNetDataset(val_samples, split='val')

y_train = np.array([train_dataset[i][1].numpy() for i in range(len(train_dataset))])
y_val = np.array([val_dataset[i][1].numpy() for i in range(len(val_dataset))])

tier1_metrics = compute_tier1_frequency_baseline(y_train, y_val, threshold=0.5)
print('=====================================================')
print('    TIER 1 BASELINE: CLASS-FREQUENCY MARGINAL PRIOR   ')
print('=====================================================')
print(f"Micro F1 Score:       {tier1_metrics['micro_f1']:.4f}")
print(f"Macro F1 Score:       {tier1_metrics['macro_f1']:.4f}")
print(f"Micro Precision:      {tier1_metrics['micro_precision']:.4f}")
print(f"Micro Recall:         {tier1_metrics['micro_recall']:.4f}")
print(f"Mean Average Precision: {tier1_metrics['mAP']:.4f}")
print('=====================================================')

### Tier 1 Findings
The Tier 1 class-frequency baseline reflects the statistical floor derived entirely from label prevalence, without visual feature analysis.

## Section 2: Tier 2 Frozen SeCo Backbone Linear Probe
We initialize the 12-band ResNet-50, freeze all feature extraction layers, and optimize only the linear classification head (`Linear(2048, 19)`) with `BCEWithLogitsLoss`.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Evaluation Device: {device}')

model = LandCoverClassifier(backbone='resnet50', num_classes=19, in_channels=12, pretrained=False)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

tier2_history = train_tier2_linear_probe(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=5,
    lr=0.001,
    device=device,
)

### Tier 2 Performance Analysis
The linear probe isolates how linearly separable the 19 land-cover classes are directly in the pretrained feature representation before full domain fine-tuning.

### Summary of Baseline Progression

### Q&A
* **What does Tier 1 measure?** The statistical baseline achievable by predicting empirical training class frequencies without imagery.
* **What does Tier 2 measure?** The linear separability of 12-band Sentinel-2 features using a frozen pretrained backbone.

### Data Analysis Key Findings
* Tier 1 baseline successfully established as the quantitative performance floor.
* Tier 2 linear probe trained the classification head cleanly with `BCEWithLogitsLoss`.
* The 12-channel stem accepts resampled multispectral Sentinel-2 tensors without runtime error.

### Insights or Next Steps
* Proceed to `03_training.ipynb` for Tier 3 end-to-end fine-tuning with differential learning rates.